In [0]:
CREATE TABLE catalog.default.t_plain (
  id INT,
  value STRING,
  event_date DATE
)
PARTITIONED BY (event_date);

CREATE TABLE catalog_commits.default.t_cc (
  id INT,
  value STRING,
  event_date DATE
)
PARTITIONED BY (event_date)
TBLPROPERTIES (
  'delta.feature.catalogManaged' = 'supported'
);

INSERT INTO catalog.default.t_plain VALUES
  (1, 'normal', DATE '2026-05-13');

INSERT INTO catalog_commits.default.t_cc VALUES
  (1, 'normal', DATE '2026-05-13');

In [0]:
%python

from pyspark.sql import functions as F

loc = "abfss://unity@westus2uc.dfs.core.windows.net/catalog/__unitystorage/catalogs/b07fd208-8482-47d4-9ebc-85cab4293acc/tables/18039d55-b172-4074-aad9-cf117e3fafa2/2026-05-14"

df = spark.createDataFrame(
    [(999, "direct_delta_path_write", "extra_value", "2026-05-14")],
    "id INT, value STRING, rogue_col STRING, event_date STRING"
).withColumn("event_date", F.to_date("event_date"))

(df.write
    .format("parquet")
    .mode("append")
    .save(loc))
